# VectorCypher Retriever

The `VectorCypherRetriever` enhances vector search with custom Cypher graph traversal. After the vector index finds matching chunks, a Cypher query runs on each match to pull in related graph context: neighboring chunks, document metadata, or connected entities.

**Learning Objectives:**
- Write a Cypher retrieval query that traverses from matched chunks to related nodes
- Use `VectorCypherRetriever` to combine semantic search with graph context
- Compare results between pure vector and vector-cypher approaches

Pure vector search returns isolated chunks. VectorCypher returns the chunk plus whatever the graph knows about its neighborhood.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

import neo4j
from lib.data_utils import get_embedder, get_llm
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.types import RetrieverResultItem
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
llm = get_llm()

print(f'Connected to Neo4j!')
print(f'LLM: {llm.model_id}')
print(f'Embedder: {embedder.model_id}')

## Define Retrieval Query

The retrieval query runs on each chunk returned by vector search. It receives the matched `node` and `score` variables and must return columns that the result formatter will process.

This query traverses from the matched chunk to its source document, then follows the `FILED` relationship to find the company that filed it. It uses `COLLECT` subqueries to gather products and risk factors linked to the chunk via `FROM_CHUNK` — keeping the results scoped to what the chunk actually mentions rather than pulling all entities for the company. This avoids cartesian products between independent fan-outs.

In [ ]:
RETRIEVAL_QUERY = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
WITH node, doc, score, company
RETURN node.text AS text,
       score,
       {document: doc.accessionNumber,
        filingType: doc.filingType,
        company: company.name,
        products: collect { MATCH (p:Product)-[:FROM_CHUNK]->(node) RETURN p.name },
        risks: collect { MATCH (r:RiskFactor)-[:FROM_CHUNK]->(node) RETURN r.name }
       } AS metadata
"""

print('Retrieval query defined!')

## Initialize VectorCypher Retriever

The `VectorCypherRetriever` takes the same vector index and embedder as the basic `VectorRetriever`, plus the custom Cypher query that will run on each match.

A custom `result_formatter` separates the chunk text (passed to the LLM as context) from the structured metadata (available for programmatic inspection). Without a formatter, the default serializes the entire record into a single string.

In [ ]:
def format_record(record: neo4j.Record) -> RetrieverResultItem:
    """Separate chunk text (content for LLM) from structured graph metadata."""
    metadata = record.get("metadata") or {}
    metadata["score"] = record.get("score")
    return RetrieverResultItem(
        content=record.get("text", ""),
        metadata=metadata,
    )

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=RETRIEVAL_QUERY,
    result_formatter=format_record,
)

print('VectorCypherRetriever initialized!')

## Search with Graph Context

Run a query with the `VectorCypherRetriever` and compare the enriched results to the plain chunk text you saw from the `VectorRetriever` in the previous notebook. Notice the additional metadata — filing info, connected companies, products, and risk factors — that the graph traversal adds to each match.

In [ ]:
query = "What are the financial risks?"

cypher_result = vector_cypher_retriever.search(query_text=query, top_k=2)

print('=== VectorCypherRetriever ===')
for i, item in enumerate(cypher_result.items, 1):
    meta = item.metadata or {}
    print(f'\n[{i}] Score: {meta.get("score", 0):.4f}')
    print(f'    Company: {meta.get("company", "N/A")}')
    print(f'    Products: {meta.get("products", [])}')
    print(f'    Risks: {meta.get("risks", [])}')
    content_preview = str(item.content)[:200]
    print(f'    Text: {content_preview}...')

## GraphRAG with Graph Context

Build a GraphRAG pipeline with the VectorCypher retriever. The LLM now receives the matched chunk along with connected entity data — companies, products, and risk factors — producing answers grounded in both unstructured text and structured knowledge graph relationships.

In [ ]:
rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)

query = "What are the key risk factors mentioned in Apple's 10-K filing?"
response = rag.search(query, retriever_config={'top_k': 5}, return_context=True)

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

print('\n\n=== Enriched Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    meta = item.metadata or {}
    print(f'\n[{i}] Score: {meta.get("score", 0):.4f} | Company: {meta.get("company", "N/A")}')
    print(f'    Products: {meta.get("products", [])} | Risks: {meta.get("risks", [])}')
    content_preview = str(item.content)[:200]
    print(f'    Text: {content_preview}...')

## Summary

The `VectorCypherRetriever` adds graph context to every vector search result:

| Approach | What the LLM Receives |
|----------|----------------------|
| **VectorRetriever** | The matched chunk text only |
| **VectorCypherRetriever** | The matched chunk text + filing metadata + connected companies, products, and risk factors |

The retrieval query uses `COLLECT` subqueries to gather products and risk factors linked to the matched chunk via `FROM_CHUNK`, keeping the context scoped to what the chunk actually mentions. A custom `result_formatter` separates the chunk text (passed to the LLM) from structured metadata (available for inspection).

Both approaches rely on vector similarity to find the initial matches. The graph traversal enriches those matches with structured knowledge — this is the core of GraphRAG.

---

**Next:** [Lab 5: Neo4j MCP Server](../Lab_5_MCP_Server/) — agent-based retrieval via MCP

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')